# Task 1 Kaggle Training Notebook

Notebook nay train **Gemma 4 12B** cho bai toan **Task 1** tren **AvaMERG + ESConv**.

Thu tu nen chay:
1. Cell 1 cai moi truong
2. Restart kernel
3. Cell 3-5 kiem tra version + login + tokenizer
4. Cell 6-7 tai va inspect du lieu
5. Cell 8 dump prompt
6. Cell 10 smoke test 1 step
7. Cell 11 train that

> Neu fail, dung lai o cell do va xem traceback truoc khi chay tiep.


In [ ]:
import os
REPO_URL = "https://github.com/QuangVoAI/multimodal-empathy-mental-health.git"
REPO_DIR = "/kaggle/working/multimodal-empathy-mental-health"
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd /kaggle/working/multimodal-empathy-mental-health
# Keep Kaggle's preinstalled torch/CUDA stack.
%pip uninstall -y datasets transformers huggingface_hub accelerate peft bitsandbytes sentencepiece tokenizers torchvision
%pip install --no-cache-dir --force-reinstall -r requirements_kaggle.txt


## Important: restart the Kaggle kernel now

Sau khi cell cai package chay xong, hay **Restart Session / Restart Kernel** roi moi chay tiep tu cell duoi day.


In [ ]:
%cd /kaggle/working/multimodal-empathy-mental-health
import transformers, huggingface_hub, torch
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("torch:", torch.__version__)


In [ ]:
from huggingface_hub import login

HF_TOKEN = "YOUR_HF_TOKEN"
login(HF_TOKEN)


In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "google/gemma-4-12B-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded:", type(tokenizer).__name__)
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)


## Import check

Cell nay chi xac nhan notebook da import duoc `run_training` truoc khi tai du lieu va chay train.


In [ ]:
from argparse import Namespace
from scripts.train_sft import run_training
print("Import ok")


In [ ]:
!bash scripts/download_avamerg.sh
!bash scripts/download_esconv.sh
!mkdir -p outputs/sft outputs/eval


In [ ]:
import json
from pathlib import Path

ava = json.loads(Path("data/raw/avamerg/train.json").read_text(encoding="utf-8"))
esc = json.loads(Path("data/raw/esconv/ESConv.json").read_text(encoding="utf-8"))
print("AvaMERG samples:", len(ava))
print("AvaMERG first keys:", list(ava[0].keys()))
print("ESConv dialogues:", len(esc))
print("ESConv first keys:", list(esc[0].keys()))


In [ ]:
from argparse import Namespace
from scripts.train_sft import run_training

base_cfg = dict(
    model_name_or_path=MODEL_ID,
    avamerg_root="data/raw/avamerg",
    avamerg_split="train",
    avamerg_text_only=True,
    esconv_json="data/raw/esconv/ESConv.json",
    output_dir="outputs/sft/debug_joint",
    max_length=1536,
    max_response_tokens=192,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    num_train_epochs=1.0,
    logging_steps=1,
    save_steps=50,
    warmup_ratio=0.03,
    max_steps=-1,
    use_lora=True,
    load_in_4bit=True,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    report_to="none",
    dump_example_prompts=True,
)

debug_args = Namespace(**base_cfg)
run_training(debug_args)


In [ ]:
!sed -n "1,200p" outputs/sft/debug_joint/example_prompts.json


In [ ]:
from argparse import Namespace

smoke_cfg = dict(base_cfg)
smoke_cfg.update({
    "output_dir": "outputs/sft/joint_smoke",
    "dump_example_prompts": False,
    "max_steps": 1,
    "logging_steps": 1,
})

smoke_args = Namespace(**smoke_cfg)
run_training(smoke_args)


In [ ]:
from argparse import Namespace

# Chay cell nay khi smoke test da on.
train_cfg = dict(base_cfg)
train_cfg.update({
    "output_dir": "outputs/sft/task1_joint_run",
    "dump_example_prompts": False,
    "gradient_accumulation_steps": 8,
    "logging_steps": 10,
    "save_steps": 100,
    "max_steps": -1,
})

train_args = Namespace(**train_cfg)
run_training(train_args)
